# pretrains_check.ipynb

Сравнение предобученных моделей для получения эмбеддингов ягод.

**Структура ноутбука:**
| # | Модель | Размер | Тип |
|---|--------|--------|-----|
| 0 | Общая настройка | — | утилиты |
| 1 | **Navec** (Natasha) | ~50 MB | статичные, русский |
| 2 | **Word2Vec RusCorpora** (gensim) | ~700 MB | статичные, русский |
| 3 | **cointegrated/rubert-tiny2** | ~100 MB | контекстуальные, русский |
| 4 | **DeepPavlov/rubert-base-cased** | ~700 MB | контекстуальные, русский |
| 5 | **paraphrase-multilingual-MiniLM-L12-v2** | ~120 MB | sentence-level, мультиязычный |
| 6 | **Дообучение** rubert-tiny2 на корпусе ягод | — | MLM fine-tuning |
| 7 | **Сравнение** всех моделей | — | PCA side-by-side |


## Блок 0 — Общая настройка и данные

In [1]:
# Базовые зависимости (обычно уже есть в .venv проекта)
# %pip install numpy matplotlib scikit-learn


In [2]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ──────────────────────────────────────
# Список ягод (оба языка — для BERT-моделей нужен контекст)
# ──────────────────────────────────────
BERRIES = [
    "клубника", "земляника", "малина", "ежевика", "черника",
    "голубика", "брусника", "клюква", "крыжовник", "смородина", "облепиха",
]

# Контекстуальные описания — для BERT-моделей (они работают лучше на фразах, чем на одном слове)
BERRY_SENTENCES = [
    "клубника — красная садовая ягода",
    "земляника — маленькая красная лесная ягода",
    "малина — красная садовая ягода костянка",
    "ежевика — тёмная лесная ягода костянка",
    "черника — синяя лесная ягода",
    "голубика — синяя лесная ягода",
    "брусника — красная кислая лесная ягода",
    "клюква — красная кислая болотная ягода",
    "крыжовник — кислая садовая ягода на кусте",
    "смородина — садовая ягода на кусте",
    "облепиха — оранжевая кислая ягода на кусте",
]

# Загружаем корпус из файла (если он есть)
try:
    with open("data/berry_corpus.json", encoding="utf-8") as f:
        corpus = json.load(f)
    print(f"Корпус загружен: {len(corpus)} документов, "
          f"{sum(len(d) for d in corpus)} слов")
except FileNotFoundError:
    corpus = None
    print("data/berry_corpus.json не найден — блок дообучения пропустить")


Корпус загружен: 10 документов, 7484 слов


In [3]:
# ──────────────────────────────────────
# Утилиты: соседи + PCA-визуализация
# ──────────────────────────────────────
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-9)
    return b @ a

def show_neighbors(name, embeddings_dict, k=3):
    """Выводит k ближайших соседей для ягоды name."""
    labels = list(embeddings_dict.keys())
    matrix = np.stack(list(embeddings_dict.values()))
    idx = labels.index(name)
    sims = cosine_sim(matrix[idx], matrix)
    order = np.argsort(-sims)
    print(f"  {name:12s} →", end=" ")
    shown = 0
    for j in order:
        if j != idx:
            print(f"{labels[j]}({sims[j]:.2f})", end="  ")
            shown += 1
            if shown >= k:
                break
    print()

def plot_pca(embeddings_dict, title, ax=None):
    """PCA 2D plot для словаря {ягода: вектор}."""
    labels = list(embeddings_dict.keys())
    matrix = np.stack(list(embeddings_dict.values()))
    xy = PCA(n_components=2).fit_transform(matrix)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(xy[:, 0], xy[:, 1], s=40)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, (xy[i, 0], xy[i, 1]), fontsize=9,
                    xytext=(4, 3), textcoords="offset points")
    ax.set_title(title, fontsize=11)
    # ax.axis("off")
    if own_fig:
        plt.tight_layout()
        plt.show()

print("Утилиты готовы.")


Утилиты готовы.


---
## Блок 1 — Navec (Natasha project)

**Что это:** компактные статичные русские векторы, обученные на художественной литературе (~12B токенов).
Словарь 500k слов, dim=300, квантизованные (~50 MB).

**Плюсы:** маленький размер, быстрая загрузка, хорошее качество для русского.
**Минусы:** статичные — одно слово = один вектор, без учёта контекста.


In [4]:
%pip install navec --quiet


Note: you may need to restart the kernel to use updated packages.


In [5]:
import urllib.request, os, pathlib
from navec import Navec

NAVEC_PATH = pathlib.Path("data/navec_hudlit.tar")
NAVEC_URL = (
    "https://storage.yandexcloud.net/natasha-navec/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

if not NAVEC_PATH.exists():
    NAVEC_PATH.parent.mkdir(exist_ok=True)
    print("Скачиваем navec (~50 MB)...")
    urllib.request.urlretrieve(NAVEC_URL, NAVEC_PATH)
    print("Готово.")

navec = Navec.load(str(NAVEC_PATH))
print(f"Navec загружен: vocab={len(navec.vocab.words)}, dim={navec.pq.dim}")


Navec загружен: vocab=500002, dim=300


In [6]:
# Получаем эмбеддинги для ягод
navec_embs = {}
missing = []
for berry in BERRIES:
    if berry in navec.vocab:
        navec_embs[berry] = np.array(navec[berry], dtype=np.float32)
    else:
        missing.append(berry)

print(f"Нашли в словаре: {len(navec_embs)}/{len(BERRIES)}")
if missing:
    print(f"Не нашли: {missing}")


Нашли в словаре: 11/11


In [7]:
# Ближайшие соседи
print("Navec — ближайшие соседи:")
for b in BERRIES:
    if b in navec_embs:
        show_neighbors(b, navec_embs)


Navec — ближайшие соседи:
  клубника     → малина(0.76)  земляника(0.73)  смородина(0.68)  
  земляника    → черника(0.76)  клубника(0.73)  малина(0.73)  
  малина       → клубника(0.76)  смородина(0.75)  черника(0.74)  
  ежевика      → черника(0.55)  земляника(0.54)  смородина(0.52)  
  черника      → брусника(0.81)  голубика(0.76)  земляника(0.76)  
  голубика     → черника(0.76)  брусника(0.74)  клюква(0.72)  
  брусника     → черника(0.81)  клюква(0.75)  голубика(0.74)  
  клюква       → брусника(0.75)  черника(0.75)  голубика(0.72)  
  крыжовник    → смородина(0.71)  малина(0.65)  клюква(0.61)  
  смородина    → малина(0.75)  облепиха(0.73)  черника(0.73)  
  облепиха     → смородина(0.73)  брусника(0.65)  черника(0.64)  


In [8]:
# PCA-визуализация
plot_pca(navec_embs, "Navec (худлит, 300d)")
plt.savefig("pictures/navec_pca.png", dpi=120, bbox_inches="tight")
print("Сохранено: pictures/navec_pca.png")


Сохранено: pictures/navec_pca.png


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\2714536359.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 2 — Word2Vec RusCorpora (gensim downloader)

**Что это:** Skip-gram Word2Vec, обученный на Russian National Corpus.
Модель `word2vec-ruscorpora-300` (~700 MB) доступна прямо через gensim.downloader.

**Особенность:** слова в этой модели хранятся с POS-тегом: `малина_NOUN`.
Нужно приписывать `_NOUN` к ягодам.


In [9]:
%pip install gensim --quiet


Note: you may need to restart the kernel to use updated packages.


In [10]:
import gensim.downloader as gensim_api

# Список доступных моделей:
# print(list(gensim_api.info()['models'].keys()))

print("Загружаем word2vec-ruscorpora-300 (~700 MB, может занять несколько минут)...")
w2v_model = gensim_api.load("word2vec-ruscorpora-300")
print(f"Загружено. Размер словаря: {len(w2v_model)}")


Загружаем word2vec-ruscorpora-300 (~700 MB, может занять несколько минут)...
Загружено. Размер словаря: 184973


In [11]:
# В этой модели слова хранятся как «малина_NOUN»
w2v_embs = {}
missing_w2v = []
for berry in BERRIES:
    key = f"{berry}_NOUN"
    if key in w2v_model:
        w2v_embs[berry] = w2v_model[key]
    elif berry in w2v_model:  # попробуем без тега
        w2v_embs[berry] = w2v_model[berry]
    else:
        missing_w2v.append(berry)

print(f"Нашли: {len(w2v_embs)}/{len(BERRIES)}")
if missing_w2v:
    print(f"Не нашли: {missing_w2v}")


Нашли: 11/11


In [12]:
# Ближайшие соседи через встроенный метод модели
print("Word2Vec — ближайшие слова к 'малина_NOUN':")
for word, sim in w2v_model.most_similar("малина_NOUN", topn=8):
    print(f"  {word}: {sim:.3f}")


Word2Vec — ближайшие слова к 'малина_NOUN':
  смородина_NOUN: 0.777
  земляника_NOUN: 0.712
  ягода_NOUN: 0.711
  клубника_NOUN: 0.702
  крыжовник_NOUN: 0.683
  черника::голубика_NOUN: 0.682
  вишня_NOUN: 0.673
  черника_NOUN: 0.665


In [13]:
# PCA ягод
print("\nWord2Vec — ближайшие соседи среди ягод:")
for b in BERRIES:
    if b in w2v_embs:
        show_neighbors(b, w2v_embs)

plot_pca(w2v_embs, "Word2Vec RusCorpora (300d)")
plt.savefig("pictures/w2v_pca.png", dpi=120, bbox_inches="tight")
print("Сохранено: pictures/w2v_pca.png")



Word2Vec — ближайшие соседи среди ягод:
  клубника     → земляника(0.72)  малина(0.70)  смородина(0.68)  
  земляника    → клубника(0.72)  малина(0.71)  черника(0.68)  
  малина       → смородина(0.78)  земляника(0.71)  клубника(0.70)  
  ежевика      → малина(0.66)  смородина(0.65)  крыжовник(0.61)  
  черника      → голубика(0.74)  брусника(0.73)  земляника(0.68)  
  голубика     → черника(0.74)  брусника(0.72)  клюква(0.67)  
  брусника     → черника(0.73)  клюква(0.73)  голубика(0.72)  
  клюква       → брусника(0.73)  голубика(0.67)  черника(0.66)  
  крыжовник    → смородина(0.72)  малина(0.68)  клубника(0.65)  
  смородина    → малина(0.78)  крыжовник(0.72)  клубника(0.68)  
  облепиха     → смородина(0.60)  малина(0.56)  голубика(0.56)  
Сохранено: pictures/w2v_pca.png


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\2714536359.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 3 — cointegrated/rubert-tiny2

**Что это:** маленький двуязычный (RU+EN) BERT от Сергея Коваля (cointegrated).
~29M параметров, dim=312. Хорошо работает на CPU, практичен для экспериментов.

**Как получаем эмбеддинг:** tokenize → forward → mean-pool по всем токенам (без [CLS]/[SEP]).
Можно подавать как одно слово, так и описательную фразу — второй вариант точнее.


In [14]:
%pip install transformers --quiet


Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
from transformers import AutoTokenizer, AutoModel

TINY_MODEL = "cointegrated/rubert-tiny2"
tiny_tok = AutoTokenizer.from_pretrained(TINY_MODEL)
tiny_model = AutoModel.from_pretrained(TINY_MODEL)
tiny_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tiny_model = tiny_model.to(device)

print(f"Модель загружена на {device}")
print(f"Параметров: {sum(p.numel() for p in tiny_model.parameters()):,}")


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена на cuda
Параметров: 29,193,768


In [16]:
@torch.no_grad()
def get_bert_embedding(texts, tokenizer, model, batch_size=32, max_len=64):
    """Mean-pooling по токенам (без CLS/SEP), поддерживает батч."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        # last_hidden_state: (B, seq_len, dim)
        hidden = out.last_hidden_state
        # маскируем padding
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb = (hidden * mask).sum(1) / mask.sum(1)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)


In [17]:
# Вариант A: подаём одно слово
embs_word = get_bert_embedding(BERRIES, tiny_tok, tiny_model)
tiny_word_embs = {b: embs_word[i] for i, b in enumerate(BERRIES)}

# Вариант B: подаём описательную фразу
embs_sent = get_bert_embedding(BERRY_SENTENCES, tiny_tok, tiny_model)
tiny_sent_embs = {b: embs_sent[i] for i, b in enumerate(BERRIES)}

print("Эмбеддинги получены.")
print(f"Dim = {embs_word.shape[1]}")


Эмбеддинги получены.
Dim = 312


In [18]:
print("RuBERT-tiny2 (одно слово) — соседи:")
for b in BERRIES:
    show_neighbors(b, tiny_word_embs)

print("\nRuBERT-tiny2 (фраза с описанием) — соседи:")
for b in BERRIES:
    show_neighbors(b, tiny_sent_embs)


RuBERT-tiny2 (одно слово) — соседи:
  клубника     → брусника(0.86)  земляника(0.77)  малина(0.72)  
  земляника    → голубика(0.89)  черника(0.79)  брусника(0.78)  
  малина       → черника(0.78)  клюква(0.74)  голубика(0.74)  
  ежевика      → клюква(0.82)  крыжовник(0.77)  черника(0.73)  
  черника      → земляника(0.79)  голубика(0.79)  малина(0.78)  
  голубика     → земляника(0.89)  черника(0.79)  малина(0.74)  
  брусника     → клубника(0.86)  земляника(0.78)  смородина(0.75)  
  клюква       → ежевика(0.82)  черника(0.77)  малина(0.74)  
  крыжовник    → ежевика(0.77)  черника(0.77)  клюква(0.74)  
  смородина    → брусника(0.75)  земляника(0.74)  малина(0.72)  
  облепиха     → земляника(0.72)  голубика(0.71)  смородина(0.71)  

RuBERT-tiny2 (фраза с описанием) — соседи:
  клубника     → малина(0.93)  брусника(0.90)  земляника(0.89)  
  земляника    → черника(0.91)  голубика(0.90)  брусника(0.90)  
  малина       → клубника(0.93)  брусника(0.89)  земляника(0.89)  
  ежевика   

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(tiny_word_embs, "rubert-tiny2 (слово)", ax=axes[0])
plot_pca(tiny_sent_embs, "rubert-tiny2 (фраза)", ax=axes[1])
plt.suptitle("cointegrated/rubert-tiny2", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/rubert_tiny_pca.png", dpi=120, bbox_inches="tight")
plt.show()
print("Сохранено: pictures/rubert_tiny_pca.png")


Сохранено: pictures/rubert_tiny_pca.png


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\66885715.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 4 — DeepPavlov/rubert-base-cased

**Что это:** полноразмерный BERT (110M параметров), обученный на русском тексте
командой DeepPavlov. dim=768.

**Когда использовать:** когда нужно качество, а не скорость. На GPU работает быстро,
на CPU каждый forward займёт несколько секунд.


In [20]:
from transformers import AutoTokenizer, AutoModel
import torch

DP_MODEL = "DeepPavlov/rubert-base-cased"
dp_tok   = AutoTokenizer.from_pretrained(DP_MODEL)
dp_model = AutoModel.from_pretrained(DP_MODEL)
dp_model.eval().to(device)

print(f"Параметров: {sum(p.numel() for p in dp_model.parameters()):,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Параметров: 177,853,440


In [21]:
# Используем ту же функцию get_bert_embedding
embs_dp_word = get_bert_embedding(BERRIES, dp_tok, dp_model)
embs_dp_sent = get_bert_embedding(BERRY_SENTENCES, dp_tok, dp_model)

dp_word_embs = {b: embs_dp_word[i] for i, b in enumerate(BERRIES)}
dp_sent_embs = {b: embs_dp_sent[i] for i, b in enumerate(BERRIES)}

print(f"Dim = {embs_dp_word.shape[1]}")


Dim = 768


In [22]:
print("DeepPavlov RuBERT (слово) — соседи:")
for b in BERRIES:
    show_neighbors(b, dp_word_embs)

print("\nDeepPavlov RuBERT (фраза) — соседи:")
for b in BERRIES:
    show_neighbors(b, dp_sent_embs)


DeepPavlov RuBERT (слово) — соседи:
  клубника     → брусника(0.82)  земляника(0.72)  малина(0.69)  
  земляника    → черника(0.91)  смородина(0.84)  голубика(0.82)  
  малина       → земляника(0.79)  черника(0.74)  смородина(0.71)  
  ежевика      → земляника(0.76)  черника(0.74)  смородина(0.73)  
  черника      → земляника(0.91)  смородина(0.81)  голубика(0.79)  
  голубика     → земляника(0.82)  черника(0.79)  брусника(0.75)  
  брусника     → клубника(0.82)  земляника(0.80)  черника(0.79)  
  клюква       → черника(0.75)  голубика(0.74)  земляника(0.72)  
  крыжовник    → брусника(0.72)  клубника(0.65)  ежевика(0.64)  
  смородина    → земляника(0.84)  черника(0.81)  голубика(0.75)  
  облепиха     → ежевика(0.68)  земляника(0.68)  черника(0.67)  

DeepPavlov RuBERT (фраза) — соседи:
  клубника     → брусника(0.93)  черника(0.90)  малина(0.89)  
  земляника    → черника(0.93)  брусника(0.91)  голубика(0.88)  
  малина       → клубника(0.89)  брусника(0.88)  черника(0.86)  
  ежеви

In [23]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(dp_word_embs, "rubert-base (слово)", ax=axes[0])
plot_pca(dp_sent_embs, "rubert-base (фраза)", ax=axes[1])
plt.suptitle("DeepPavlov/rubert-base-cased", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/rubert_base_pca.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\2794023341.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 5 — Sentence Transformers (multilingual)

**Что это:** модели, специально обученные на семантическую близость предложений
(contrastive learning на парах переводов / NLI). Возвращают один вектор на текст —
не нужно делать mean-pooling самому.

**Модели:**
- `paraphrase-multilingual-MiniLM-L12-v2` (~120 MB, 50 языков, быстрая)
- `paraphrase-multilingual-mpnet-base-v2` (~430 MB, качество лучше)

Для ягод лучше всего подавать описательные фразы.


In [24]:
%pip install sentence-transformers --quiet


Note: you may need to restart the kernel to use updated packages.


In [25]:
from sentence_transformers import SentenceTransformer

ST_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(ST_MODEL, device=str(device))
print(f"Модель загружена: {ST_MODEL}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Модель загружена: paraphrase-multilingual-MiniLM-L12-v2


In [26]:
# Вариант A: одно слово
st_word_embs_arr = st_model.encode(BERRIES, normalize_embeddings=True)
st_word_embs = {b: st_word_embs_arr[i] for i, b in enumerate(BERRIES)}

# Вариант B: фраза с описанием
st_sent_embs_arr = st_model.encode(BERRY_SENTENCES, normalize_embeddings=True)
st_sent_embs = {b: st_sent_embs_arr[i] for i, b in enumerate(BERRIES)}

print(f"Dim = {st_word_embs_arr.shape[1]}")


Dim = 384


In [27]:
print("SentenceTransformer (слово) — соседи:")
for b in BERRIES:
    show_neighbors(b, st_word_embs)

print("\nSentenceTransformer (фраза) — соседи:")
for b in BERRIES:
    show_neighbors(b, st_sent_embs)


SentenceTransformer (слово) — соседи:
  клубника     → ежевика(0.44)  малина(0.44)  крыжовник(0.43)  
  земляника    → брусника(0.81)  крыжовник(0.81)  черника(0.81)  
  малина       → черника(0.90)  ежевика(0.87)  брусника(0.86)  
  ежевика      → черника(0.88)  брусника(0.88)  малина(0.87)  
  черника      → брусника(0.95)  облепиха(0.90)  малина(0.90)  
  голубика     → крыжовник(0.78)  облепиха(0.59)  брусника(0.59)  
  брусника     → черника(0.95)  облепиха(0.92)  крыжовник(0.89)  
  клюква       → черника(0.77)  брусника(0.77)  облепиха(0.73)  
  крыжовник    → брусника(0.89)  облепиха(0.87)  черника(0.87)  
  смородина    → облепиха(0.73)  черника(0.71)  брусника(0.71)  
  облепиха     → брусника(0.92)  черника(0.90)  крыжовник(0.87)  

SentenceTransformer (фраза) — соседи:
  клубника     → брусника(0.89)  малина(0.87)  крыжовник(0.84)  
  земляника    → брусника(0.94)  клюква(0.89)  малина(0.87)  
  малина       → брусника(0.89)  крыжовник(0.89)  смородина(0.89)  
  ежевика    

In [28]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(st_word_embs, "SentenceTransformer (слово)", ax=axes[0])
plot_pca(st_sent_embs, "SentenceTransformer (фраза)", ax=axes[1])
plt.suptitle("paraphrase-multilingual-MiniLM-L12-v2", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/st_pca.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\4281475567.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 6 — Дообучение rubert-tiny2 на корпусе ягод (MLM)

**Идея:** берём уже загруженный `rubert-tiny2` и дообучаем его на
wiki-корпусе через Masked Language Modeling.

После дообучения эмбеддинги должны лучше отражать специфику ягодной лексики —
"черника" и "голубика" сблизятся, "малина" и "клубника" тоже.

**Что используем:** `DataCollatorForLanguageModeling` (15% масок случайно)
+ `Trainer` из HuggingFace. Достаточно 3–5 эпох.

> Требует наличия `data/berry_corpus.json` из `embeddings_check.ipynb`.


In [29]:
assert corpus is not None, "Нужен data/berry_corpus.json — запустите embeddings_check.ipynb"

from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
from torch.utils.data import Dataset as TorchDataset

# Загружаем заново модель для MLM (нужен AutoModelForMaskedLM, не AutoModel)
FT_MODEL = "cointegrated/rubert-tiny2"
ft_tok = AutoTokenizer.from_pretrained(FT_MODEL)
ft_model_mlm = AutoModelForMaskedLM.from_pretrained(FT_MODEL)
print("MLM-версия модели загружена")


Loading weights:   0%|          | 0/58 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: cointegrated/rubert-tiny2
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MLM-версия модели загружена


In [30]:
# Подготовка датасета: весь корпус → список строк → токенизация
flat_texts = [" ".join(doc) for doc in corpus]

class BerryMLMDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
    def __len__(self):
        return self.encodings["input_ids"].shape[0]
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}

mlm_dataset = BerryMLMDataset(flat_texts, ft_tok)
print(f"Датасет: {len(mlm_dataset)} примеров")


Датасет: 10 примеров


In [31]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=ft_tok,
    mlm=True,
    mlm_probability=0.15,
)

training_args = TrainingArguments(
    output_dir="data/rubert-tiny2-berry",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none"          # отключаем wandb/mlflow
)

trainer = Trainer(
    model=ft_model_mlm,
    args=training_args,
    train_dataset=mlm_dataset,
    data_collator=data_collator,
)

print("Начинаем дообучение...")
trainer.train()
print("Готово!")


Начинаем дообучение...


Step,Training Loss


Готово!


In [32]:
# Сохраняем дообученные веса
ft_model_mlm.save_pretrained("data/rubert-tiny2-berry")
ft_tok.save_pretrained("data/rubert-tiny2-berry")
print("Дообученная модель сохранена в data/rubert-tiny2-berry/")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Дообученная модель сохранена в data/rubert-tiny2-berry/


In [33]:
# Сравниваем эмбеддинги ДО и ПОСЛЕ дообучения
# Базовую модель уже считали выше (tiny_sent_embs / tiny_word_embs)
# Теперь берём encoder из дообученной MLM-модели

@torch.no_grad()
def get_mlm_embeddings(texts, tok, model_mlm, device="cpu"):
    """Извлекаем base-encoder из AutoModelForMaskedLM и делаем mean-pool."""
    enc = tok(texts, padding=True, truncation=True,
               max_length=64, return_tensors="pt").to(device)
    # у AutoModelForMaskedLM есть .bert или .roberta — ищем base-модель
    base = getattr(model_mlm, "bert", None) or getattr(model_mlm, "roberta", None)
    if base is None:
        # Универсальный fallback: первый nn.Module с именем, содержащим 'encoder'
        for name, mod in model_mlm.named_children():
            if "encoder" in name or "bert" in name:
                base = mod
                break
    out = base(**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    return emb.cpu().numpy()

ft_model_mlm.eval()
ft_embs_arr = get_mlm_embeddings(BERRY_SENTENCES, ft_tok, ft_model_mlm, device)
ft_embs = {b: ft_embs_arr[i] for i, b in enumerate(BERRIES)}

print("Эмбеддинги после дообучения получены.")


Эмбеддинги после дообучения получены.


In [34]:
print("После fine-tuning (фразы) — соседи:")
for b in BERRIES:
    show_neighbors(b, ft_embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(tiny_sent_embs, "rubert-tiny2 ДО (фраза)", ax=axes[0])
plot_pca(ft_embs, "rubert-tiny2 ПОСЛЕ MLM (фраза)", ax=axes[1])
plt.suptitle("Дообучение rubert-tiny2 на корпусе ягод", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/finetuned_pca.png", dpi=120, bbox_inches="tight")
plt.show()


После fine-tuning (фразы) — соседи:
  клубника     → малина(0.93)  брусника(0.90)  земляника(0.90)  
  земляника    → черника(0.91)  голубика(0.90)  брусника(0.90)  
  малина       → клубника(0.93)  брусника(0.90)  земляника(0.89)  
  ежевика      → черника(0.88)  земляника(0.88)  брусника(0.87)  
  черника      → голубика(0.96)  земляника(0.91)  брусника(0.90)  
  голубика     → черника(0.96)  земляника(0.90)  брусника(0.88)  
  брусника     → клюква(0.94)  клубника(0.90)  земляника(0.90)  
  клюква       → брусника(0.94)  земляника(0.87)  малина(0.87)  
  крыжовник    → смородина(0.90)  облепиха(0.89)  брусника(0.86)  
  смородина    → крыжовник(0.90)  облепиха(0.84)  клубника(0.84)  
  облепиха     → крыжовник(0.89)  клюква(0.86)  смородина(0.84)  


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\3182245299.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Блок 7 — Сравнение всех моделей

Собираем всё в одну сетку PCA-графиков.


In [35]:
all_models = {}

# Блок 1
if 'navec_embs' in dir() or 'navec_embs' in globals():
    all_models["Navec (300d)"] = navec_embs

# Блок 2
if 'w2v_embs' in dir() or 'w2v_embs' in globals():
    all_models["Word2Vec RusCorpora (300d)"] = w2v_embs

# Блок 3
if 'tiny_sent_embs' in dir() or 'tiny_sent_embs' in globals():
    all_models["rubert-tiny2 (фраза)"] = tiny_sent_embs

# Блок 4
if 'dp_sent_embs' in dir() or 'dp_sent_embs' in globals():
    all_models["rubert-base (фраза)"] = dp_sent_embs

# Блок 5
if 'st_sent_embs' in dir() or 'st_sent_embs' in globals():
    all_models["SentenceTransformer (фраза)"] = st_sent_embs

# Блок 6
if 'ft_embs' in dir() or 'ft_embs' in globals():
    all_models["rubert-tiny2 + MLM fine-tune"] = ft_embs

print(f"Доступно моделей для сравнения: {len(all_models)}")
for name in all_models:
    print(f"  • {name}")


Доступно моделей для сравнения: 6
  • Navec (300d)
  • Word2Vec RusCorpora (300d)
  • rubert-tiny2 (фраза)
  • rubert-base (фраза)
  • SentenceTransformer (фраза)
  • rubert-tiny2 + MLM fine-tune


In [36]:
n = len(all_models)
if n == 0:
    print("Запусти хотя бы один блок выше, чтобы получить эмбеддинги.")
else:
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 6))
    axes = np.array(axes).flatten()

    for i, (name, embs) in enumerate(all_models.items()):
        plot_pca(embs, name, ax=axes[i])

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Сравнение предобученных моделей — ягоды (PCA 2D)", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("pictures/all_models_pca.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Сохранено: pictures/all_models_pca.png")


Сохранено: pictures/all_models_pca.png


C:\Users\uuu10\AppData\Local\Temp\ipykernel_31620\3906732069.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
# Числовое сравнение: средняя косинусная близость «правильных пар»
# (ягоды, которые должны быть близко по смыслу)
expected_pairs = [
    ("клубника",  "земляника"),  # обе красные, садовые
    ("малина",    "ежевика"),    # обе костянки
    ("черника",   "голубика"),   # обе синие лесные
    ("брусника",  "клюква"),     # обе красные кислые
    ("крыжовник", "смородина"),  # обе на кусте, садовые
]

print(f"{'Модель':<40} {'Среднее sim правильных пар':>28}")
print("-" * 70)
for name, embs in all_models.items():
    sims = []
    for a, b in expected_pairs:
        if a in embs and b in embs:
            va = embs[a] / (np.linalg.norm(embs[a]) + 1e-9)
            vb = embs[b] / (np.linalg.norm(embs[b]) + 1e-9)
            sims.append(float(va @ vb))
    if sims:
        print(f"{name:<40} {np.mean(sims):>28.4f}")


Модель                                     Среднее sim правильных пар
----------------------------------------------------------------------
Navec (300d)                                                   0.6960
Word2Vec RusCorpora (300d)                                     0.7142
rubert-tiny2 (фраза)                                           0.9108
rubert-base (фраза)                                            0.8705
SentenceTransformer (фраза)                                    0.8775
rubert-tiny2 + MLM fine-tune                                   0.9125
